# NB_lab_test_trends
**Silver → Gold | CareSync Health Network** (per STTM v1.0)

* **Target:** `adb_caresync.gold.lab_test_trends`
* **Grain:** one row per test type (test_name) per calendar month
* **Driving table:** `adb_caresync.silver.lab_results` grouped by test_name, year/month of test_date
* **Joins:** LEFT JOIN `calendar` on date_key = date_format(date_trunc('month', test_date), 'yyyyMMdd')
* **Window:** LAG(abnormal_rate_pct) partitioned by test_name, ordered by year, month (month-over-month trend)
* **Load:** FULL — CREATE OR REPLACE TABLE ... AS SELECT

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS adb_caresync.gold;

In [0]:
%sql
CREATE OR REPLACE TABLE adb_caresync.gold.lab_test_trends
COMMENT 'Gold: one row per test_name per month with abnormal/critical rates and month-over-month trend (STTM v1.0)'
AS
WITH monthly AS (
  SELECT
    test_name,
    YEAR(test_date)                                                       AS year,
    MONTH(test_date)                                                      AS month,
    CAST(DATE_FORMAT(DATE_TRUNC('MONTH', test_date), 'yyyyMMdd') AS INT)  AS month_date_key,
    COUNT(*)                                                              AS total_tests,
    COUNT_IF(result_status = 'Abnormal')                                  AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')                                  AS critical_test_count
  FROM adb_caresync.silver.lab_results
  GROUP BY test_name, YEAR(test_date), MONTH(test_date),
           CAST(DATE_FORMAT(DATE_TRUNC('MONTH', test_date), 'yyyyMMdd') AS INT)
),
rates AS (
  SELECT
    m.*,
    CAST(m.abnormal_test_count * 100.0 / m.total_tests AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(m.critical_test_count * 100.0 / m.total_tests AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly m
),
with_lag AS (
  SELECT
    r.*,
    LAG(r.abnormal_rate_pct) OVER (PARTITION BY r.test_name ORDER BY r.year, r.month) AS prev_month_abnormal_rate_pct
  FROM rates r
)
SELECT
  w.test_name,
  CAST(w.year AS INT)                                                     AS year,
  CAST(w.month AS INT)                                                    AS month,
  cal.month_name,
  cal.quarter_name,
  CAST(w.total_tests AS INT)                                              AS total_tests,
  CAST(w.abnormal_test_count AS INT)                                      AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)                                      AS critical_test_count,
  w.abnormal_rate_pct,
  w.critical_rate_pct,
  CAST(w.prev_month_abnormal_rate_pct AS DECIMAL(5,2))                    AS prev_month_abnormal_rate_pct,
  CAST(w.abnormal_rate_pct - w.prev_month_abnormal_rate_pct AS DECIMAL(6,2)) AS abnormal_rate_change_pct
FROM with_lag w
LEFT JOIN adb_caresync.silver.calendar cal ON cal.date_key = w.month_date_key;

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM adb_caresync.gold.lab_test_trends)                                  AS gold_rows,
  (SELECT COUNT(DISTINCT test_name, year, month) FROM adb_caresync.gold.lab_test_trends)    AS gold_distinct_grain,
  (SELECT SUM(total_tests) FROM adb_caresync.gold.lab_test_trends)                          AS gold_total_tests,
  (SELECT COUNT(*) FROM adb_caresync.silver.lab_results)                                    AS silver_lab_results,
  (SELECT COUNT(*) FROM adb_caresync.gold.lab_test_trends WHERE month_name IS NULL)         AS rows_missing_calendar;

In [0]:
record_count = spark.sql("SELECT COUNT(*) AS record_count FROM adb_caresync.gold.lab_test_trends").first()["record_count"]



dbutils.notebook.exit(record_count)